In [2]:
import pandas as pd
import numpy as np
import os

base_dir = '../data/raw/mm-fit/w00/'

# 1. 正确读取标签文件（没有表头）
label_file = os.path.join(base_dir, 'w00_labels.csv')
# 手动指定列名：起始帧, 结束帧, 标签ID, 动作名
df_label = pd.read_csv(label_file, header=None, names=['start_frame', 'end_frame', 'label_id', 'activity'])
print("=== 标签文件 (w00_labels.csv) ===")
print(f"形状: {df_label.shape}")
print(f"所有动作:\n{df_label['activity'].value_counts()}")
display(df_label.head(10))

# 2. 读取传感器数据 (.npy文件)
# 读取左手手表加速度数据
acc_file = os.path.join(base_dir, 'w00_sw_l_acc.npy')
acc_data = np.load(acc_file)
print(f"\n=== 左手手表加速度 (w00_sw_l_acc.npy) ===")
print(f"形状: {acc_data.shape} (总帧数, 轴数)")
print(f"前5帧:\n{acc_data[:5]}")

# 读取左手手表陀螺仪数据
gyr_file = os.path.join(base_dir, 'w00_sw_l_gyr.npy')
gyr_data = np.load(gyr_file)
print(f"\n=== 左手手表陀螺仪 (w00_sw_l_gyr.npy) ===")
print(f"形状: {gyr_data.shape}")
print(f"前5帧:\n{gyr_data[:5]}")

=== 标签文件 (w00_labels.csv) ===
形状: (30, 4)
所有动作:
activity
squats                     3
pushups                    3
dumbbell_shoulder_press    3
lunges                     3
dumbbell_rows              3
situps                     3
tricep_extensions          3
bicep_curls                3
lateral_shoulder_raises    3
jumping_jacks              3
Name: count, dtype: int64


,start_frame,end_frame,label_id,activity
0,4040,4500,10,squats
1,5435,5928,10,squats
2,7200,7685,10,squats
3,8770,9197,11,pushups
4,11160,11586,10,pushups
5,13620,14000,10,pushups
6,15034,15545,10,dumbbell_shoulder_press
7,17570,18127,10,dumbbell_shoulder_press
8,20482,20908,9,dumbbell_shoulder_press
9,22147,22990,10,lunges



=== 左手手表加速度 (w00_sw_l_acc.npy) ===
形状: (220297, 5) (总帧数, 轴数)
前5帧:
[[ 4.05000000e+03  1.56279087e+12  9.77168000e+00 -5.07744130e-01
   7.66645770e+00]
 [ 4.05000000e+03  1.56279087e+12  1.00087870e+01 -3.56857930e-01
   8.39214900e+00]
 [ 4.05000000e+03  1.56279087e+12  1.00974030e+01 -3.44882820e-01
   9.02203900e+00]
 [ 4.05100000e+03  1.56279087e+12  1.02291290e+01 -1.19750980e-02
   9.00527400e+00]
 [ 4.05100000e+03  1.56279087e+12  1.06171220e+01  3.20932630e-01
   8.87594300e+00]]

=== 左手手表陀螺仪 (w00_sw_l_gyr.npy) ===
形状: (220297, 5)
前5帧:
[[ 4.05000000e+03  1.56279087e+12 -1.83259500e-01 -2.07694100e-01
  -5.31452540e-01]
 [ 4.05000000e+03  1.56279087e+12 -1.97920260e-01 -1.75929130e-01
  -4.77696450e-01]
 [ 4.05000000e+03  1.56279087e+12 -1.11177430e-01 -1.46607610e-01
  -4.22718580e-01]
 [ 4.05100000e+03  1.56279087e+12  8.55211000e-03 -1.08733974e-01
  -3.62853830e-01]
 [ 4.05100000e+03  1.56279087e+12  1.38055490e-01 -8.30776400e-02
  -2.71224050e-01]]


In [3]:
import numpy as np
import pandas as pd
import os
from glob import glob

base_dir = '../data/raw/mm-fit/'
session_dirs = sorted(glob(os.path.join(base_dir, 'w*')))

all_records = []
label_map = {}   # 动作名 → 数字标签
current_label = 0

for session_dir in session_dirs:
    session_name = os.path.basename(session_dir)
    
    # 读取传感器数据（左手手表）
    acc_file = os.path.join(session_dir, f'{session_name}_sw_l_acc.npy')
    gyr_file = os.path.join(session_dir, f'{session_name}_sw_l_gyr.npy')
    if not os.path.exists(acc_file) or not os.path.exists(gyr_file):
        continue
    
    acc_data = np.load(acc_file)   # (T, 5)
    gyr_data = np.load(gyr_file)   # (T, 5)
    
    # 读取标签文件
    label_file = os.path.join(session_dir, f'{session_name}_labels.csv')
    df_label = pd.read_csv(label_file, header=None, 
                           names=['start_frame', 'end_frame', 'label_id', 'activity'])
    
    for _, row in df_label.iterrows():
        start_f = int(row['start_frame'])
        end_f = int(row['end_frame'])
        activity = row['activity']
        
        # 根据帧号切片（注意：帧号从 0 开始？标签中的帧号是绝对帧号）
        if start_f < 0 or end_f >= acc_data.shape[0] or end_f <= start_f:
            continue
        
        acc_seg = acc_data[start_f:end_f+1, 2:5]  # 跳过前两列时间戳
        gyr_seg = gyr_data[start_f:end_f+1, 2:5]
        
        # 映射数字标签
        if activity not in label_map:
            label_map[activity] = current_label
            current_label += 1
        lbl = label_map[activity]
        
        # 构建 DataFrame
        df_temp = pd.DataFrame({
            'acc_x': acc_seg[:, 0],
            'acc_y': acc_seg[:, 1],
            'acc_z': acc_seg[:, 2],
            'gyro_x': gyr_seg[:, 0],
            'gyro_y': gyr_seg[:, 1],
            'gyro_z': gyr_seg[:, 2],
            'label': lbl,
            'subject_id': session_name   # 暂时用课程名作为 subject_id
        })
        all_records.append(df_temp)

if all_records:
    df_final = pd.concat(all_records, ignore_index=True)
    print("✅ 提取完成")
    print(f"总行数：{len(df_final)}")
    print(f"\n动作映射：")
    for name, lid in sorted(label_map.items(), key=lambda x: x[1]):
        print(f"  {lid} -> {name}")
    print(f"\n各动作数据量：")
    print(df_final['label'].value_counts().sort_index())
    
    os.makedirs('../data/processed/', exist_ok=True)
    df_final.to_csv('../data/processed/mmfit_processed.csv', index=False)
    print("\n✅ 已保存到 data/processed/mmfit_processed.csv")
else:
    print("❌ 没有提取到任何数据")

✅ 提取完成
总行数：373774

动作映射：
  0 -> squats
  1 -> pushups
  2 -> dumbbell_shoulder_press
  3 -> lunges
  4 -> dumbbell_rows
  5 -> situps
  6 -> tricep_extensions
  7 -> bicep_curls
  8 -> lateral_shoulder_raises
  9 -> jumping_jacks

各动作数据量：
label
0    38940
1    31975
2    39988
3    52398
4    33497
5    48017
6    37888
7    34582
8    38890
9    17599
Name: count, dtype: int64

✅ 已保存到 data/processed/mmfit_processed.csv
